# N09 · Prefix Cache：为什么“看起来一样”的 prompt 可能不命中？


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

Prefix cache 的想法很简单：多个请求共享相同前缀时，前缀的 KV cache 可以复用，避免重复 prefill。但工程上最容易踩坑的是：人眼看起来相同，不代表 tokenized prefix 相同。

本节目标：理解 prefix cache 命中的条件、RadixAttention 的直觉、cache hit rate 怎么解释，以及 repeated-prefix benchmark 怎么设计。


## 学习地图与版本说明（截至 2026-04-30）

本节的核心是“缓存命中按 token 前缀，不按人类语义”。两个 prompt 看起来意思一样，只要 chat template、空格、换行、标点、文档排序、tokenizer 版本或系统提示不同，都可能变成不同 token 序列而无法命中。线上 RAG/Agent 服务中，这类小差异经常让 prefix cache hit rate 远低于预期。

版本上，本教程参考 SGLang RadixAttention 论文与官方文档，以及 vLLM automatic prefix caching / metrics 文档。RadixAttention 用 radix tree 组织可复用前缀，适合多轮对话、few-shot 模板、RAG 固定系统提示等场景；vLLM/SGLang 的具体指标名和默认开关可能随版本变化，实验前要查看当前安装版本的 CLI 与 metrics 文档。

学完本节，你应该能构造 repeated-prefix benchmark，并能解释为什么 hit rate 高不一定 TTFT 一定大幅下降：剩余 uncached token、队列、decode、网络、后处理仍可能占主导。你也应该能提出 cache-aware routing、模板规范化、稳定文档排序等工程修复，而不是只说“打开 prefix cache”。


## 1. 命中的第一条件：tokenized prefix 一致

Prefix cache 通常按 token 序列匹配，不是按“语义相似”匹配。下面这些都可能导致 miss：

- 中文冒号 `：` vs 英文冒号 `:`。
- 末尾多一个空格或换行。
- chat template 改变 system/user/assistant 标签。
- tokenizer 版本改变。
- RAG 文档排序改变。
- 随机插入 timestamp/request id。

所以 benchmark 必须保证 repeated prefix 字节和 token 都一致。


In [ ]:
prefixes = [
    "系统提示: 你是一个严谨的助手。
",
    "系统提示: 你是一个严谨的助手。
",
    "系统提示：你是一个严谨的助手。
",  # 中文冒号
    "系统提示: 你是一个严谨的助手。 
", # 多空格
]

seen = set()
rows = []
for i, p in enumerate(prefixes, 1):
    hit = p in seen
    rows.append({"请求": i, "repr(prefix)": repr(p), "byte_len": len(p.encode()), "cache_hit": hit})
    seen.add(p)

pd.DataFrame(rows)


## 2. RadixAttention 的直觉

SGLang 论文提出 RadixAttention：把多个请求的 token prefix 组织成类似 radix tree / compact prefix tree 的结构，查找最长公共前缀并复用 KV cache。

直觉：

```text
A: system + few-shot examples + question1
B: system + few-shot examples + question2
C: system + different examples + question3
```

A 和 B 可以共享 `system + few-shot examples` 的 KV；C 只能共享 `system` 或更短前缀。

这对 agent、RAG、多轮对话、固定工具 schema 特别重要，因为它们有大量重复系统提示和固定上下文。


In [ ]:
# 一个极简 trie，用字符模拟 token prefix cache。真实系统按 token/block 管理。
class TrieNode:
    def __init__(self):
        self.children = {}
        self.count = 0

root = TrieNode()

def insert_and_match(tokens):
    node = root
    matched = 0
    for tok in tokens:
        if tok in node.children:
            node = node.children[tok]
            matched += 1
        else:
            break
    node = root
    for tok in tokens:
        node = node.children.setdefault(tok, TrieNode())
        node.count += 1
    return matched

requests = [
    "SYSTEM 工具A 工具B 问题1".split(),
    "SYSTEM 工具A 工具B 问题2".split(),
    "SYSTEM 工具A 工具C 问题3".split(),
]
for req in requests:
    print(req, "matched_prefix_tokens=", insert_and_match(req))


## 3. cache hit rate 怎么解释？

Cache hit rate 高不一定代表整体快，低也不一定代表系统差。要结合：

- 命中的 prefix 有多长？命中 10 tokens 和命中 10k tokens 价值不同。
- 命中发生在 GPU cache 还是 host/offload cache？
- prefill 是否本来就很短？短 prompt 命中收益小。
- cache 管理是否造成额外开销？
- 多副本 serving 时，请求是否被路由到有缓存的 worker？

实践指标：`prefix_cache_hits/queries`、TTFT p50/p95、prompt_tokens、queue time、每个 worker 的 cache 使用率。


## 4. repeated-prefix benchmark 的正确设计

最小设计：

1. 固定一个长 system prefix。
2. 生成 N 个请求，每个请求只改变 suffix。
3. 先 warmup，让 prefix 进入 cache。
4. 对比 cache enabled vs disabled，或首轮 vs 后续轮。
5. 记录 TTFT、hit rate、prompt length、请求路由。

与本课程连接：L08 的 `bench_repeated_prefix.py` 就是这个实验的入口。


In [ ]:
def synthetic_ttft(prefix_tokens, suffix_tokens, hit, prefill_ms_per_token=0.08, overhead_ms=30):
    computed_tokens = suffix_tokens if hit else prefix_tokens + suffix_tokens
    return overhead_ms + computed_tokens * prefill_ms_per_token

rows = []
for prefix_tokens in [128, 1024, 8192]:
    for hit in [False, True]:
        rows.append({"prefix_tokens": prefix_tokens, "cache_hit": hit, "TTFT_ms": synthetic_ttft(prefix_tokens, 64, hit)})

display(pd.DataFrame(rows).round(1))


## 5. 企业面试/工程判断痛点题（带答案）

### 题 1：两个 prompt 语义一样，prefix cache 会命中吗？

**答案解析：** 不一定。cache 按 token prefix 匹配，不按语义匹配。标点、空格、模板差异都可能 miss。

### 题 2：cache hit rate 80%，为什么 TTFT 没明显下降？

**答案解析：** 可能命中的 prefix 很短、prompt 本来很短、queue time 主导、命中在慢速层级、或 benchmark 没隔离其他变量。

### 题 3：多副本服务为什么需要 cache-aware routing？

**答案解析：** 如果请求被随机发到不同 worker，即使某个 worker 有 prefix cache，后续请求也可能落到没有缓存的 worker，导致命中率下降。

### 题 4：RAG 场景最容易破坏 prefix cache 的是什么？

**答案解析：** 文档排序、chunk 内容、metadata、timestamp、用户特定上下文变化。固定系统提示能共享，但 RAG 上下文常变化。

### 题 5：如何证明 prefix cache 真的有效？

**答案解析：** 固定长前缀，控制 suffix，比较 cache enabled/disabled 或首轮/后续轮；同时记录 hit rate 和 TTFT，不能只看单次请求。


## 参考资料

- SGLang paper / RadixAttention: https://arxiv.org/abs/2312.07104
- SGLang documentation: https://docs.sglang.io/index.html
- SGLang production metrics: https://docs.sglang.io/docs/references/production_metrics
- vLLM metrics and prefix cache counters: https://docs.vllm.ai/en/stable/design/metrics/
